<a href="https://colab.research.google.com/github/williamlidberg/Analyses-of-Environmental-Data-2/blob/main/modules/module_10/Assignment_10_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating a web map
Notebooks are great for developing and GIS softwares like arcgis/QGis are nice for exploring data. But your ability to share your work is limited, at best you can print a PDF like a caveman. That is where web mapping comes in. Web mapping brings a level of accessibility and insight that traditional maps simply can't match.

\
Let's start by creating a very basic map. These lines of code will create a basemap and save it as a html file.

In [ ]:
import folium

m = folium.Map()
m.save('/content/map.html')

### Assignment 1
Download the html file to your computer and open it with your browser.

The map is based on OpenStreetMap. Like in previous exercises we can make it look cooler by using darkmode. We can also set a start location and zoom level. Note that the right zoom start level will depend on your screen size. Smaller screens can fit less map.

In [ ]:
m = folium.Map(location = [60.663, 16.885], zoom_start=10) # Gävle, I am doing some science here.
folium.TileLayer('Cartodb dark_matter').add_to(m) # Sets Tile Theme to (Dark Theme)
m.save('/content/cool_map.html')

Now you can add a basemap and select an area where the map should focus. Now we can add some data to the map. This example uses harvest blocks from the Swedish forest agency. Start by downloading the data.

In [ ]:
!wget https://geodpags.skogsstyrelsen.se/geodataport/data/sksAvverkAnm21.zip
!unzip /content/sksAvverkAnm21.zip

You can import the shapefile using geopandas.

In [ ]:
import geopandas as gpd

harvests = gpd.read_file('/content/sksAvverkAnm21.shp')
harvests.head()

Combine the dataframe with folium to create a html map. You can also change settings for the colors of the polygons and thicknes of the lines.

In [ ]:
m = folium.Map(location = [61.526, 16.413], zoom_start=9) # Gävle
folium.TileLayer('cartodbpositron').add_to(m) # Sets Tile Theme to gray

choropleth = folium.Choropleth(
    geo_data=harvests,  # Harvest sites geodataframe
    fill_color='red',
    line_weight=0.1,
    line_color='white',
).add_to(m)

m.save('/content/harvest_map.html')

You can also subsample the data to show the most recent applications.

In [ ]:
recent_harvest_applications = harvests[harvests['ArendeAr'] > 2022]

m = folium.Map(location=[61.345, 16.514], zoom_start=9, tiles='cartodbpositron') # Sets Tile Theme to Gray

folium.Choropleth(
    geo_data=recent_harvest_applications,
    name='Choropleth',
    data=recent_harvest_applications,
    columns=['OBJECTID', 'AnmaldHa'], # OBJECTID is the ID column of the
    key_on="feature.properties.OBJECTID",
    fill_color='YlGnBu',
    fill_opacity=1,
    line_opacity=0.2,
    legend_name='Harvest area in ha',
    smooth_factor=0,
    control=False  # Add this line to hide the default legend
).add_to(m)

# Add legend to the map
folium.LayerControl().add_to(m)

m.save('/content/recent_harvest_map.html')


### Task 2
Make a html file for a webmap from wetlands in Gävleborg
https://geodata.naturvardsverket.se/nedladdning/VMI/ursprunglig_digitalisering/X_Gavleborg_VMI.zip

In [ ]:
!wget https://geodata.naturvardsverket.se/nedladdning/VMI/ursprunglig_digitalisering/X_Gavleborg_VMI.zip
!unzip /content/X_Gavleborg_VMI.zip -d /content/X_Gavleborg_VMI

# Heatmaps
A popular way to display large amounts of data is to create a heatmap. For this example you will use location of houses in Gävle. The data is avalible at [Sveriges Dataportal](https://www.dataportal.se/datasets/574_1040/baskarta-byggnader#ref=?p=4&q=g%C3%A4vle&s=2&t=20&f=&rt=dataset%24esterms_IndependentDataService%24esterms_ServedByDataService&c=false).




In [ ]:
from urllib.request import urlretrieve
url = ('https://catalog.gavle.se/store/1/resource/48')
filename = '/content/baskarta-adresser.zip' # you need to adjust this path on your own computer if you are using anaconda.
urlretrieve(url, filename)
!unzip /content/baskarta-adresser.zip -d /content/baskarta_adresser

Inspect the data and give each building a uniq ID.

In [ ]:
import geopandas as gpd
buildings = gpd.read_file('/content/baskarta_adresser/baskarta_adress.shp')
# Add a new column named 'ID' with sequential IDs starting from 1
buildings['id'] = range(1, len(buildings) + 1)
buildings

In [ ]:
from folium.plugins import HeatMap


m = folium.Map(location=[buildings['WGS84_LAT'].mean(), buildings['WGS84_LONG'].mean()], zoom_start=9)
heat_data = [[row['WGS84_LAT'], row['WGS84_LONG']] for index, row in buildings.iterrows()]
HeatMap(heat_data).add_to(m)

map_title = "Heatmap of buildings"
title_html = f'<h1 style="position:absolute;z-index:100000;left:40vw" >{map_title}</h1>'
m.get_root().html.add_child(folium.Element(title_html))
# Save the map
m.save('/content/index.html') # note that I renamed it index.html for this to work with github.


Zoom in and out on the heatmap and watch how it dynamically aggregates the data.


### Popups
We can also add popups with information from the attribute table to each polygon in a webmap. Here is an example from the Krycklan catchment.

In [ ]:
from urllib.request import urlretrieve
url = ('https://gis.krycklan.se/Krycklan_SurveyCatchments.zip')
filename = '/content/Krycklan_SurveyCatchments.zip' # you need to adjust this path on your own computer if you are using anaconda.
urlretrieve(url, filename)
!unzip -o /content/Krycklan_SurveyCatchments.zip -d /content/Krycklan_SurveyCatchments

In [ ]:
import geopandas as gpd

shp_path = "/content/Krycklan_SurveyCatchments/SurveyCatchments.shp"
gdf = gpd.read_file(shp_path)
gdf = gdf.to_crs(epsg=4326)
gdf.head()

In [ ]:
import folium
import branca.colormap as cm

# Create color scale
colormap = cm.linear.YlOrRd_09.scale(
    gdf["Area"].min(),
    gdf["Area"].max()
)
colormap.caption = "Catchment area (km²)"

# Map extent
bounds = gdf.total_bounds

# Create map
m = folium.Map(
    location=[
        (bounds[1] + bounds[3]) / 2,
        (bounds[0] + bounds[2]) / 2
    ],
    zoom_start=12,
    tiles="CartoDB positron"
)

# Add polygons with data-driven colors
folium.GeoJson(
    gdf,
    name="Krycklan Catchments",
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["Area"]),
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.6,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["GRIDCODE"],
        aliases=["Catchment:"]
    ),
    popup=folium.GeoJsonPopup(
        fields=["GRIDCODE", "Area"],
        aliases=["Catchment:", "Area (km²):"],
        localize=True
    )
).add_to(m)

# Add legend
colormap.add_to(m)

folium.LayerControl().add_to(m)

m.save("/content/index.html")


## Task 3 Publish the web map.. on the web
So far we have created a html file that can be opened in a browser. While this file can be shared (and in many cases better than a pdf) it is not yet a webmap. For that we need a way to host it online. There are a few different way of doing that. There are multiple services which can host html files for you or you can host them yourself from a computer or server. Read the instructions bellow and pick a way to host your html file to the world. The GitHub option is easier and will be enough for you to pass the assignment. The other alternative is for you to explore if you like.

Include the URL somewhere in the notebook so I can have a look at your webmap.

## Using GitHub
For this option you need to create a GiHub account. GitHub is a developer tool that can be used to devlop and share projects. Once you have created a github account you can create follow these [instructions](https://docs.github.com/en/pages/quickstart) to create a new repository and activate github pages. The key parts of the instructions are:


1.   Create a new repository with default settings and name it username.github.io where username is your github username. for me this would be [williamlidberg.github.io](williamlidberg.github.io)
3.   On the next page you will be given the option to create a new file or upload an existing file. Upload the heatmap html file created above.
2.   Wait 10 minutes and navigate to username.github.io with username replaced with your own. For me this would be https://williamlidberg.github.io/ it will display error 404 if you dont wait a bit.
## Using google cloud
Most cloud providers offer some kind of free or low cost tier for hosting webpages. In this tutorial we will use google cloud since you already have a google account set up for google colab.
1. Go to [google cloud](https://www.google.com/aclk?sa=L&ai=DChsSEwiVoPHrmIGSAxXxV0ECHai4FyYYACICCAEQAhoCd3M&ae=2&co=1&ase=2&gclid=CjwKCAiAjojLBhAlEiwAcjhrDkIzFVAxHvHWNObRSW6ig89wrPab8mimHiTs_1gr53V_p0jmNAwRdxoCl0AQAvD_BwE&cce=2&category=acrcp_v1_71&sig=AOD64_1LrkMq8CTvWJaCe2EvNJZbCvQ1Rw&q&nis=4&adurl&ved=2ahUKEwj08errmIGSAxX-YPEDHUPtO9cQ0Qx6BAgOEAE) and click the button "get started for free". It will ask for a credit card which might seem fishy but it's only used to verify users, no money will be charged unless you convert your trial account to a real account. Once inside you will be greeted with a pop up window with some boxes. Feel free to close it.
2. On the top left on your screen you should see "My First Project", a search bar and some small icons. Click the icon "activate cloud shell" and accept the promts. Once the terminal is open we will use it to create a storage location "a bucket" and assign it to "My First Project". For this you need to find the ID of your "My First Project" by clicking on it on the top left. second you need to decide a name for your storage bucket. My ID was project-1659f630-ad16-499b-98a and I named my bucket gullbyn but you can pick a more relevant name for your bucket. If the name is taken you will get this error: HTTPError 409: The requested bucket name is not available.

```
gcloud storage buckets create gs://gullbyn --project=project-1659f630-ad16-499b-98a --location=europe-north1
```
3. Open the bucket for viewing using this command (change ID and name):

```
gcloud storage buckets add-iam-policy-binding gs://gullbyn \
    --member=allUsers \
    --role=roles/storage.objectViewer
```
4. Now we need to locate the bucket using the UI and uppload the index.html file to your bucket. Start by using the search bar and search for buckets. This should take you to your bucket. Click it and select upload files. Find your index.html file creates in colab and upload it.


5. Finally, create a webpage using an index.html file creates in google colab.
```
gcloud storage buckets update gs://gullbyn \
    --web-main-page-suffix=index.html
```

The webmap is now accecible on https://storage.googleapis.com/gullbyn/index.html

You will have to change gullbyn to whatever name you used to create the bucket.



# If you want to try hosting your own server
This section is not part of the course and you can stop here if you wish. Bellow are instructions on how to set up a webmap using local hardware. Windows 10 reached end of life and the hardware requrements for windows 11 means that there are many old capable machines that cannot be upgraded to windows 11. These old machines are exellent candiates to install linux on and use to host your own projects.


There are several options for web server software available, depending on your operating system. One of the popular options is Apache HTTPD, which is available for all major platforms, another is Microsoft IIS which is available for Windows operating systems. However, the rising star and also my personal favorite is NGINX (pronounced Engine X), which is available for all major platforms and several smaller platforms.

The installation instructions will vary slightly depending on your operating system so I have divided them up based on your OS below. The default configuration should work perfectly for simple sites such as these.

For all operating systems, after nginx is up and running your site will be visible locally to you if you visit http://localhost in your web browser (http://localhost:8080 on MacOS), others will have to enter your public IP-address in the address field of their browser instead of localhost, you can find your public IP by visiting i.a. https://whatismyip.org/ (make sure you are not connected to a VPN), this IP may change if you restart your router. If you get a "secure connection failed" error, make sure that the browser has not replaced the http:// with http**s**://.

Its important to note that there are other things to keep in mind if you plan on hosting your own web server in the future. Proper configuration and keeping the operating system and web server software up-to date is absolutely crutial to limit the risk of malevolent individuals breaching your server and using it for their own purposes. In these cases, you should read up more closely on security best practices and how to enable HTTPS for your website (which is almost a must for any serious website these days).

#### Windows

Download the latest release of NGINX from https://nginx.org/en/download.html (https://nginx.org/download/nginx-1.26.0.zip at the time of writing) and extract the downloaded .zip-file.

Within the extracted directory, you should see an executable named nginx.exe and a couple of directories. The *conf* directory contains the configuration files and the ***html*** directory contains the files that will be visible to those visiting your server in a web browser.

Start nginx by navigating to the extracted folder in a terminal and typing: start nginx.

Windows should alert you that NGINX wants to communicate with the internet, press allow to let it through the firewall.

#### Linux

Nginx is almost certainly available in your distributions repositories, so you should install it from there.

1. **Debian/Ubuntu/Linux Mint/PopOS:** *sudo apt install nginx*
2. **RedHat/Fedora/CentOS/Rocky/Alma:** *sudo dnf in nginx*
3. **Arch/Manjaro:** *sudo pacman -S nginx*
4. **openSUSE:** *sudo zypper in nginx*

After installation, enable it using: *sudo systemctl enable --now nginx*.

The default configuration might vary slightly between distributions, but in most cases you will want to place your html files in the ***/var/www/html***-folder.

If you have a firewall installed, you will have to allow connections over HTTP, this will vary slightly depending on your firewall software.

* **For ufw (default on ubuntu):** *sudo ufw allow http*
* **For firewalld (default on RedHat and Fedora):** *sudo firewall-cmd --permanent --zone=public --add-service=http*, followed by *sudo firewall-cmd --reload*

#### MacOS

The easiest way to install NGINX on mac is by using homebrew. If you dont have homebrew installed you can follow the instructions here to get it: https://docs.brew.sh/Installation

Then install by typing (in the terminal): *brew install nginx*, it will take a while to complete.

Now you can enable nginx by typing: *brew services start nginx*.

You will want to place your html files in the ***/opt/homebrew/var/www***-folder

If you want to enable connections from outside your home network, you need to let them through the firewall. You can do that by executing (in the terminal): *sudo /usr/libexec/ApplicationFirewall/socketfilterfw --add /usr/local/bin/nginx*

#### Opening up your web server to the outside world

If you set up your own web server, you will probably need to also open the correct port in your router. This is not necessary if you are not using a router (i.e. the computer is connected via cable directly to the wall or fiber converter). The instructions in this case will vary widely depending on the make and model of your router, so it is not feasible to provide detailed instructions here. However, the basic steps are:

1. Assign a static internal IP-address to the computer that is running your web server.
2. Forward port 80 (or port 8080 for MacOS) to the static IP you set in the first step.

Port forwarding is explained here for Asus routers: https://www.asus.com/support/faq/1037906/, and here for TP-Link routers: https://www.tp-link.com/se/support/faq/1379/. These are probably the two most common consumer-grade routers.